## Verify files in relevant volume

In [0]:

directory = "/Volumes/bronze_dev/census_bureau/census_bureau_raw"
file_path_xlsx_unprocessed = f"{directory}/co-est2025-pop.xlsx"
file_path_xlsx = f"{directory}/co-est2025-pop processed.xlsx"
file_path_csv = file_path_xlsx.replace("xlsx","csv")


display(dbutils.fs.ls(directory))

## Pandas read from xlsx = working


In [0]:
import pandas as pd

df = pd.read_excel(file_path_xlsx)

print(df.head())

## spark read from processed xlsx

Read Processed data

In [0]:
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.functions import col
from pyspark.sql.functions import col, when

# Load from sheet
df = spark.read.format("excel") \
    .load(file_path_xlsx_unprocessed)

# Rename columns
new_cols = ["county", "pop_base_2020","pop_est_2020","pop_est_2021","pop_est_2022","pop_est_2023","pop_est_2024","pop_est_2025"]
df = df.toDF(*new_cols)

# Remove the header rows, the row for US total, and the footer rows
df = df.withColumn("row_id", monotonically_increasing_id())
df = df.filter("row_id >= 5").filter("row_id < 3145").drop("row_id")

display(df)

In [0]:
df = spark.read.format("excel") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("dataAddress", "'CO-EST2025-POP'!$A$5:$H$3149") \
    .load(file_path_xlsx_unprocessed)

#new_cols = ["county", "pop_base_2020","pop_est_2020","pop_est_2021","pop_est_2022","pop_est_2023","pop_est_2024","pop_est_2025"]
#df = df.toDF(*new_cols)

display(df)

In [0]:
from pyspark.sql.functions import col, split, regexp_replace

clean_col = regexp_replace(col("county"), r"^\.", "")

df_clean = df.withColumn("county_name", split(clean_col, ", ").getItem(0)) \
             .withColumn("state_name", split(clean_col, ", ").getItem(1)) \
             .drop("county")
display(df_clean)

In [0]:
# Reorder columns to put county and state first
df_final = df_clean.select(
    "county_name",
    "state_name",
    *[c for c in df_clean.columns if c not in ["county_name", "state_name"]]
)
display(df_final)

In [0]:
df_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_dev.census_bureau.county_population")

In [0]:
%sql
SELECT * FROM bronze_dev.census_bureau.county_population LIMIT 10;